# Full Autonomous Run — ITQ Bottle Cap Collector

**Purpose:** Run the complete autonomous ball collection system.

**Prerequisites:**
1. Basket calibrated (run `00_calibrate_basket.ipynb`)
2. Ball colors calibrated (run `01_calibrate_colors.ipynb`)
3. All hardware tested (run `robot_tests/robot_diagnostics.ipynb`)

**What this does:**
- Initializes all hardware (camera, motors, arm)
- Runs state machine loop
- Searches for balls
- Approaches and collects balls
- Returns to basket and deposits
- Repeats until time limit or manual stop

## 1. Setup

In [ ]:
import sys
import os
import time
import yaml
import cv2
import numpy as np
from IPython.display import display, Image
import ipywidgets.widgets as widgets
from jetbot import Robot

# Add project root to path
project_root = '/home/steve/Notebooks/Projects/AI & Robotics Hackathon Berlin/ITQ_HackLab_Team_2'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import modules
from perception.ball_detector import BallDetector
from perception.obstacle_detector import ObstacleDetector
from perception.basket_detector import BasketDetector
from control.state_machine import StateMachine
from control.navigator import Navigator
from hardware.camera import CameraController
from hardware.arm import ArmController

print("✓ Modules imported")

## 2. Load Configuration

In [ ]:
config_path = os.path.join(project_root, 'config.yaml')

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("✓ Configuration loaded")
print(f"  Arena: {config['arena']['width_cm']}x{config['arena']['height_cm']} cm")
print(f"  Camera: {config['camera']['width']}x{config['camera']['height']}")
print(f"  Max speed: {config['motors']['max_speed']}")

## 3. Initialize Hardware

In [ ]:
print("Initializing hardware...")

# Robot (motors)
robot = Robot()
print("✓ Robot initialized")

# Camera
camera_ctrl = CameraController(config)
if not camera_ctrl.initialize():
    raise Exception("Camera initialization failed!")
camera_ctrl.center()
camera_ctrl.look_down()  # Look down for ground-level balls
print("✓ Camera initialized")

# Arm
arm = ArmController(config)
arm.home()
print("✓ Arm initialized")

print("\n✓ All hardware ready!")

## 4. Initialize Perception

In [ ]:
print("Initializing perception modules...")

ball_detector = BallDetector(config)
print("✓ Ball detector ready")

obstacle_detector = ObstacleDetector(config)
print("✓ Obstacle detector ready")

basket_detector = BasketDetector(config)
print("✓ Basket detector ready")

# Check if basket is calibrated
if config['basket']['size_px'] is None:
    print("\n⚠ WARNING: Basket not calibrated!")
    print("Run 00_calibrate_basket.ipynb first.")
else:
    print(f"✓ Basket calibrated (size: {config['basket']['size_px']:.0f} px²)")

## 5. Initialize Control

In [ ]:
print("Initializing control modules...")

state_machine = StateMachine(config)
print("✓ State machine ready")

navigator = Navigator(robot, config)
navigator.set_frame_size(config['camera']['width'], config['camera']['height'])
print("✓ Navigator ready")

print("\n✓ System ready for autonomous run!")

## 6. Test Perception (Optional)

In [ ]:
# Capture test frame
test_frame = camera_ctrl.read()

if test_frame is not None:
    # Test ball detection
    balls = ball_detector.detect(test_frame)
    print(f"Balls detected: {len(balls)}")
    for ball in balls:
        color, centroid, distance, area = ball
        print(f"  - {color}: {distance:.0f} cm")
    
    # Test obstacle detection
    obstacle = obstacle_detector.detect_combined(test_frame)
    print(f"\nBoundary: {obstacle['boundary_detected']}")
    print(f"Obstacle: {obstacle['obstacle_detected']}")
    
    # Test basket detection
    basket = basket_detector.detect(test_frame)
    print(f"\nBasket found: {basket['basket_found']}")
    if basket['basket_found']:
        print(f"  Distance: {basket['distance']:.0f} cm")
        print(f"  Bearing: {basket['bearing']:.0f}°")
    
    # Draw overlays
    overlay = ball_detector.draw_detections(test_frame, balls)
    overlay = obstacle_detector.draw_detections(overlay, obstacle)
    overlay = basket_detector.draw_detection(overlay, basket)
    
    # Display
    _, buffer = cv2.imencode('.jpg', overlay)
    display(Image(data=buffer.tobytes()))
else:
    print("ERROR: Could not capture frame")

## 7. Main Loop

**IMPORTANT:** This will start autonomous operation. Robot will move!

**To stop:** Click the "STOP" button or interrupt the kernel.

In [ ]:
# Control variables
running = False
start_time = None
time_limit = 300  # 5 minutes

# Widgets
start_button = widgets.Button(description="START", button_style='success')
stop_button = widgets.Button(description="STOP", button_style='danger')
status_output = widgets.Output()

def on_start_clicked(b):
    global running, start_time
    running = True
    start_time = time.time()
    with status_output:
        print("\n" + "="*50)
        print("AUTONOMOUS RUN STARTED")
        print("="*50)

def on_stop_clicked(b):
    global running
    running = False
    navigator.stop()
    with status_output:
        print("\n" + "="*50)
        print("STOPPED BY USER")
        print("="*50)

start_button.on_click(on_start_clicked)
stop_button.on_click(on_stop_clicked)

display(widgets.HBox([start_button, stop_button]))
display(status_output)

In [ ]:
# Main control loop
loop_count = 0
last_status_time = time.time()

try:
    while running:
        loop_count += 1
        current_time = time.time()
        elapsed = current_time - start_time
        
        # Check time limit
        if elapsed > time_limit:
            with status_output:
                print("\nTime limit reached!")
            break
        
        # Capture frame
        frame = camera_ctrl.read()
        if frame is None:
            with status_output:
                print("Frame capture failed!")
            time.sleep(0.1)
            continue
        
        # Run perception
        balls = ball_detector.detect(frame)
        validated_balls = ball_detector.validate_detection(balls)
        obstacle = obstacle_detector.detect_combined(frame)
        basket = basket_detector.detect(frame)
        
        # Prepare perception data
        perception_data = {
            'balls': validated_balls,
            'obstacle': obstacle,
            'basket': basket
        }
        
        # Update state machine
        response = state_machine.update(perception_data)
        
        # Execute action
        action = response['action']
        target = response['target']
        
        if action == 'collect':
            # Execute pickup sequence
            navigator.stop()
            arm.pickup_sequence()
        
        elif action == 'deposit':
            # Execute deposit sequence
            navigator.stop()
            arm.deposit_sequence()
        
        else:
            # Normal navigation
            navigator.execute_action(action, target)
        
        # Print status every 2 seconds
        if current_time - last_status_time > 2.0:
            with status_output:
                status = state_machine.get_status()
                print(f"\n[{elapsed:.0f}s] State: {status['state']} | Balls: {status['balls_collected']}")
                print(f"  {response['message']}")
            last_status_time = current_time
        
        # Small delay to prevent CPU overload
        time.sleep(0.05)  # 20 Hz loop

except KeyboardInterrupt:
    with status_output:
        print("\nInterrupted by user")

finally:
    # Stop everything
    running = False
    navigator.stop()
    arm.home()
    
    # Final status
    with status_output:
        print("\n" + "="*50)
        print("RUN COMPLETE")
        print("="*50)
        final_status = state_machine.get_status()
        print(f"Balls collected: {final_status['balls_collected']}")
        print(f"Total time: {elapsed:.1f} seconds")
        print(f"Loop iterations: {loop_count}")

## 8. Cleanup

In [ ]:
# Release resources
camera_ctrl.release()
robot.stop()
arm.home()

print("✓ Cleanup complete")